##### Copyright 2025。

# Gemma 2 使用 LangChain 和 Groq 呼叫函數

本教學示範如何使用 [Groq Cloud](https://console.groq.com/) 上託管的 `gemma-2-9b-it` 模型以及 **LangChain 整合** 來執行 **函數呼叫**。
<table align="left"> <td>    <a target="_blank" href="https://colab.research.google.com/github/google-gemini/gemma-cookbook/blob/main/Gemma/[Gemma_2]Function_Calling_with_Groq_Langchain.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td>
</table>

In [1]:
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

## 設定

您將需要：
- A [Groq API 鍵](https://console.groq.com/keys)
- Python 3.10+
- Colab secrets 已啟用以安全地存取金鑰。

In [2]:
!pip install --quiet groq langchain langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.7/126.7 kB 1.8 MB/s eta 0:00:00


In [3]:
import os
from google.colab import userdata

# Load your Groq API key from Colab secrets
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

## 定義外部工具（函數）

我們定義一個獲取天氣資訊的範例函數。

In [27]:
from langchain.tools import Tool
from langchain_groq import ChatGroq
from langchain.agents import initialize_agent
import re

# Define a single-input weather tool
def get_current_weather_tool(input_str: str) -> str:
    """Get the current weather in a location."""
    # Naive parsing: look for location and optional unit
    match = re.search(r"in (\w+)", input_str.lower())
    unit_match = re.search(r"(celsius|fahrenheit)", input_str.lower())

    location = match.group(1) if match else "unknown"
    unit = unit_match.group(1) if unit_match else "fahrenheit"
    return f"Final Answer: The weather in {location.title()} is 72° {unit.title()} and sunny."


## 在 Groq 上使用 Gemma 2 初始化 LangChain Agent

我們將透過 LangChain 使用`gemma-2-9b-it`並綁定我們的工具。

In [28]:
get_current_weather = Tool(
    name="get_current_weather",
    func=get_current_weather_tool,
    description="Get the current weather by passing a string like 'What's the weather in Paris in Celsius?'",
    return_direct=True
)

# Initialize LLM
llm = ChatGroq(model="gemma-2-9b-it", temperature=0)

# Initialize the agent (no need to bind tools for this setup)
agent = initialize_agent(
    tools=[get_current_weather],
    llm=llm,
    agent="zero-shot-react-description",
    verbose=True
)


## 執行範例提示

我們現在執行自然語言prompt，它應該觸發函數呼叫。

In [29]:
# Example usage
response = agent.run("What's the weather in Tokyo in Fahrenheit?")
print(response)



> Entering new AgentExecutor chain...
Thought: I need to get the current weather in Tokyo, and the user wants the temperature in Fahrenheit. I should use the get_current_weather function to get the current weather.

Action: get_current_weather
Action Input: What's the weather in Tokyo in Fahrenheit?
Observation: Final Answer: The weather in Tokyo is 72° Fahrenheit and sunny.


> Finished chain.
Final Answer: The weather in Tokyo is 72° Fahrenheit and sunny.


## 結論

在這篇notebook中，您學習如何：
- 使用 LangChain 透過 Groq 連接Gemma 2 (`gemma-2-9b-it`)
- 定義並綁定外部工具（函數）
- 使用自然prompts 透過 LangChain Agents 觸發函數調用

這是一個強大的構建塊，用於將 APIs 和邏輯整合到 LLM 工作流程中。